# Data Quality Scorecard

Assess dataset health across completeness, validity, consistency, and timeliness.

In [ ]:
import sys
sys.path.insert(0, '/home/user/nyc_data')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Try to import toolkit
try:
    from socrata_toolkit.governance import compute_quality_score
except ImportError:
    print("Note: Using sample data")

print("✓ Environment initialized")

In [ ]:
def generate_quality_data():
    """Generate sample quality scores for key datasets."""
    datasets = [
        ('inspection', 94, 0.92, 0.96, 0.98, 0.87),
        ('violations', 91, 0.89, 0.94, 0.95, 0.84),
        ('ramp_progress', 87, 0.85, 0.91, 0.93, 0.79),
        ('ramp_complaints', 86, 0.82, 0.90, 0.92, 0.78),
        ('street_permits', 83, 0.80, 0.87, 0.88, 0.75),
        ('dismissals', 85, 0.83, 0.89, 0.90, 0.77),
    ]
    
    data = []
    for name, overall, completeness, validity, consistency, timeliness in datasets:
        data.append({
            'dataset': name,
            'overall_score': overall,
            'completeness': completeness,
            'validity': validity,
            'consistency': consistency,
            'timeliness': timeliness
        })
    
    return pd.DataFrame(data)

df_quality = generate_quality_data()
print("✓ Loaded quality scores")

## Overall Score by Dataset

In [ ]:
# Color-code by quality level
colors = []
for score in df_quality['overall_score']:
    if score >= 90:
        colors.append('rgb(0, 204, 150)')  # Green
    elif score >= 80:
        colors.append('rgb(255, 165, 0)')  # Orange
    else:
        colors.append('rgb(239, 85, 59)')  # Red

fig = go.Figure(data=[
    go.Bar(
        x=df_quality['dataset'],
        y=df_quality['overall_score'],
        marker_color=colors,
        text=[f"{score:.0f}" for score in df_quality['overall_score']],
        textposition='outside',
    )
])

fig.update_layout(
    title='Overall Data Quality Score',
    xaxis_title='Dataset',
    yaxis_title='Quality Score (0-100)',
    yaxis=dict(range=[0, 105]),
    height=400,
    showlegend=False
)

fig.show()

## Quality Dimensions

In [ ]:
# Radar chart for quality dimensions
fig = go.Figure()

for idx, row in df_quality.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row['completeness'], row['validity'], row['consistency'], 
           row['timeliness'], row['overall_score']],
        theta=['Completeness', 'Validity', 'Consistency', 'Timeliness', 'Overall'],
        fill='toself',
        name=row['dataset'],
        opacity=0.6
    ))

fig.update_layout(
    title='Quality Dimensions by Dataset',
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    height=500
)

fig.show()

## Dimension Breakdown

In [ ]:
# Heatmap of quality dimensions
dimensions = df_quality[['dataset', 'completeness', 'validity', 'consistency', 'timeliness']].set_index('dataset')

fig = px.imshow(
    dimensions.T,
    labels=dict(x='Dataset', y='Dimension', color='Score'),
    color_continuous_scale='RdYlGn',
    zmin=0,
    zmax=100,
    title='Quality Dimensions Heatmap',
    height=400
)

fig.update_yaxes(tickangle=0)
fig.show()

## Quality Summary Table

In [ ]:
# Display quality scores as table
display_df = df_quality.copy()
for col in ['overall_score', 'completeness', 'validity', 'consistency', 'timeliness']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.0f}")

print("\nData Quality Assessment:")
print(display_df.to_string(index=False))
print("\nScoring:")
print("  90+ = Excellent (green)")
print("  80-89 = Good (orange)")
print("  <80 = Fair (red)")